In [4]:
!pip install -q transformers datasets

In [6]:
from datasets import load_dataset
import pandas as pd
from IPython.display import display

# Streaming se poora dataset download karna zaroori nahi.
dataset = load_dataset(
    "rathachat/writing-prompts",
    split="train",
    streaming=True
)

records = list(dataset.take(5))

print("Dataset columns:", list(records[0].keys()))

# Dataset mein prompt wali column identify karo.
prompt_column = next(
    (col for col in ["prompt", "source", "instruction"]
     if col in records[0]),
    None
)

if prompt_column is None:
    raise ValueError(
        f"Prompt column identify nahi hui. Columns: {list(records[0].keys())}"
    )

prompts = [str(row[prompt_column]).strip() for row in records]

for i, prompt in enumerate(prompts, start=1):
    print(f"\n{'=' * 65}")
    print(f"PROMPT {i}")
    print(prompt)

README.md:   0%|          | 0.00/604 [00:00<?, ?B/s]

PROMPT 1

You 've finally managed to discover the secret to immortality . Suddenly , Death appears before you , hands you a business card , and says , `` When you realize living forever sucks , call this number , I 've got a job offer for you . ''

PROMPT 2

The moon is actually a giant egg , and it has just started to hatch .

PROMPT 3

You find a rip in time walking through the alleys . You enter it to find yourself on a metal table with surgical instruments on a chair next to you .

PROMPT 4

For years in your youth the same imaginary character appears in your dreams , you are good friends . Years later , when adult , you meet her in real life , she clearly recognises you and tries to avoid you , and you want answers .

PROMPT 5

You glance at your watch 10:34 am , roughly 10 seconds later your plane explodes over the Pacific Ocean . Your eyes open as you jolt awake . The familiar hum of the planes engine remains . Checking your watch it is 9:35



In [7]:
import torch
from transformers import pipeline, set_seed

set_seed(42)

generator = pipeline(
    task="text-generation",
    model="openai-community/gpt2",
    device=0 if torch.cuda.is_available() else -1
)

generator.tokenizer.pad_token = generator.tokenizer.eos_token

print("Model loaded!")
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Model loaded!
Device: GPU


In [9]:
results = []

for i, prompt in enumerate(prompts, start=1):
    # GPT-2 ki context limit ke andar input rakho.
    token_ids = generator.tokenizer.encode(
        prompt,
        add_special_tokens=False
    )

    if len(token_ids) > 800:
        raise ValueError(
            f"Prompt {i} bohat lamba hai. Is prompt ko shorten karna hoga."
        )

    output = generator(
        prompt,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        top_k=50,
        repetition_penalty=1.1,
        num_return_sequences=1,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    completion = output[0]["generated_text"].strip()

    results.append({
        "Prompt Number": i,
        "Prompt": prompt,
        "Generated Completion": completion,
        "Manual Evaluation": ""
    })

    print(f"\n{'=' * 75}")
    print(f"STORY {i}")
    print(f"\nPROMPT:\n{prompt}")
    print(f"\nGENERATED COMPLETION:\n{completion}")

results_df = pd.DataFrame(results)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



STORY 1

PROMPT:
You 've finally managed to discover the secret to immortality . Suddenly , Death appears before you , hands you a business card , and says , `` When you realize living forever sucks , call this number , I 've got a job offer for you . ''

GENERATED COMPLETION:
"
The phone rings in response to an unexpected request from his lover. His voice is raised above that of one who was recently asked by her mother about life-changing experiences, he explains it as something she would do if there were no other options: [[She's looking at him like...'']] Her smile fades but soon disappears under some more disturbing circumstances - The man makes sure not to say anything until after dying (or perhaps even later) or when they are ready again.[/quote]

 [citation needed] This revelation has been bothering me since my first attempt with our relationship almost two years ago! What have we done wrong? Is someone really so far beyond their means right now?! If your answer turns out to be

[transformers] Both `max_new_tokens` (=180) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



STORY 2

PROMPT:
The moon is actually a giant egg , and it has just started to hatch .

GENERATED COMPLETION:
The male, called " Tawny ", also known as the Dragon's Egg or even some other name for eggs.
 (and) He doesn't look like you'll find him anywhere in this game but on his chest where he can be found next door with an extra set of clothes that will make your character taller if wearing them all day long while still getting out-of-"unnecessary" clothing before they get too far into space! You don -re not required to wear these items either...but when looking at their heads after speaking to any characters who are carrying those outfits then I think we know what exactly tl;dr: If two NPCs walk over one another during combat so close together no matter how many times someone tries to kill eachother between battles/fights there isn`t much going wrong except maybe having more enemies than people need because every single time fighting gets easier because


[transformers] Both `max_new_tokens` (=180) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



STORY 3

PROMPT:
You find a rip in time walking through the alleys . You enter it to find yourself on a metal table with surgical instruments on a chair next to you .

GENERATED COMPLETION:
The staff member is holding an old manual typewriter. He's got some notes for your questions, and he takes out his pen as well - like there was nothing left of anything that could be taken from him when they first brought this up about what happened inside us... Then I'm sitting here looking at these two people who have had their brains removed by someone else so we don't know if something went wrong or not; my wife asks me why do any men take off shoes? And she says "just because"..."so many women walk around naked all day long". My husband goes back into bed trying very hard not even noticing her being dressed differently," Mrs. Pezzino said while pointing straight ahead."The last one took away both feet", Mr Pizzino replied once more before getting down again ."My son has also been having proble

[transformers] Both `max_new_tokens` (=180) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



STORY 4

PROMPT:
For years in your youth the same imaginary character appears in your dreams , you are good friends . Years later , when adult , you meet her in real life , she clearly recognises you and tries to avoid you , and you want answers .

GENERATED COMPLETION:
You decide that it is time for a new love interest. She meets with him again at school because he has been so rude ; however, after his death from cancer they have no contact on social media :
 The relationship between them continues as usual! At first glance their looks seem very normal but soon begin to become quite different: They both look like highschool students (and sometimes older) too much of an embarrassment - even if neither really wants to be here anymore ! I know my girl doesn't wanna get out there alone all day long now 's what bothers me more than anything :) It's probably not fun ever having such close relationships  ㅋㅋ My self esteem seems pretty low right? " Well then can we just talk about something 

In [10]:
output_path = "/kaggle/working/story_completions.csv"

results_df.to_csv(output_path, index=False)

display(results_df)
print(f"\nResults saved: {output_path}")

,Prompt Number,Prompt,Generated Completion,Manual Evaluation
0,1,You 've finally managed to discover the secret...,"""\nThe phone rings in response to an unexpecte...",
1,2,"The moon is actually a giant egg , and it has ...","The male, called "" Tawny "", also known as the ...",
2,3,You find a rip in time walking through the all...,The staff member is holding an old manual type...,
3,4,For years in your youth the same imaginary cha...,You decide that it is time for a new love inte...,
4,5,"You glance at your watch 10:34 am , roughly 10...",am : 7 miles. You see that on time in a few mi...,



Results saved: /kaggle/working/story_completions.csv
